# Reto de riesgo de cola del S&P 500 — Notebook de referencia para el profesor

Este notebook descarga la tabla actual de componentes del S&P 500 de Wikipedia y los precios diarios
de **cierre ajustado (Adjusted Close)** de Yahoo Finance, calcula la asimetría muestral corregida por sesgo
y el exceso de curtosis, aplica las reglas de elegibilidad de la competición y exporta la clasificación de referencia.

**Fechas de análisis:** del 02-01-2024 al 31-12-2025, ambas inclusive.  
**Competición:** del 08-10-2026 a las 14:00 al 12-10-2026 a las 23:59, Europe/Madrid; cinco intentos.

Flujo de trabajo: configuración → universo congelado → calendario bursátil → descargas con caché →
rentabilidades entre sesiones consecutivas → elegibilidad → puntuación percentil equiponderada →
exportaciones para auditoría.

**Solo para el profesor:** las salidas revelan los tickers ganadores. No distribuyas este notebook ejecutado
ni la clasificación de referencia entre los estudiantes antes de que finalice la competición.

## Reproducibilidad y limitaciones metodológicas

- Aplicar los componentes actuales del índice a un periodo anterior introduce sesgo de supervivencia
  y de selección de componentes. Se trata de un análisis retrospectivo con fines docentes, no de un
  backtest point-in-time ni de una predicción. No se utilizan observaciones futuras de rentabilidad en el cálculo.
- Los ajustes del proveedor y la composición de Wikipedia pueden cambiar. Se conservan instantáneas locales
  de los datos de entrada, hashes SHA-256, parámetros, versiones de paquetes y marcas temporales de descarga.
  Volver a ejecutar el notebook con los mismos datos almacenados en caché reproduce los cálculos; descargarlos
  de nuevo puede modificar los resultados. Utiliza un nuevo directorio de salida para una nueva instantánea
  de la competición.
- Un cuarto momento elevado puede estar dominado por muy pocas observaciones. Una asimetría negativa
  no implica por sí sola mayores pérdidas potenciales. La equiponderación es una convención pedagógica,
  no un modelo de riesgo estimado. No es necesario asumir normalidad para construir el ranking.
- No elimines rentabilidades extremas válidas ni corrijas precios de forma silenciosa. Revisa los movimientos
  inusualmente grandes y las anomalías del proveedor antes de congelar el conjunto de datos de puntuación.
- Yahoo Finance es una fuente de datos accesible con fines educativos, no un proveedor institucional validado.
  Comprueba sus condiciones de uso antes de redistribuir los datos.


## 1. Entorno y configuración

Selecciona el intérprete de Python `.venv-tail-risk` en VS Code. Si utilizas otro entorno,
instala las dependencias mediante la celda opcional siguiente y reinicia el kernel.
No se utilizan credenciales ni archivos `.env` del proyecto. Solo se necesita acceso a Internet para
crear las instantáneas de datos de entrada o volver a intentar explícitamente alguna descarga.


In [ ]:
# Descomenta únicamente si faltan dependencias en el kernel seleccionado.
# %pip install numpy pandas scipy requests lxml yfinance exchange-calendars ipykernel

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from importlib.metadata import version
from io import StringIO
from pathlib import Path
import json
import math
import platform
import time

import exchange_calendars as xcals
import numpy as np
import pandas as pd
import requests
from scipy.stats import kurtosis, skew
import yfinance as yf
from IPython.display import display

START = pd.Timestamp('2024-01-02')
END = pd.Timestamp('2025-12-31')
MIN_COVERAGE = 0.90
KURTOSIS_WEIGHT = 0.50
SKEWNESS_WEIGHT = 0.50
BATCH_SIZE = 30
DOWNLOAD_ATTEMPTS = 3
# Actívalo explícitamente para reintentar símbolos fallidos. Los símbolos correctos almacenados en caché permanecen congelados.
RETRY_FAILED_DOWNLOADS = False
WIKIPEDIA_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'docs' else cwd
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'competitions' / 'tail_risk_2024_2025'
CACHE_DIR = OUTPUT_DIR / 'price_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def file_hash(path: Path) -> str:
    return sha256(path.read_bytes()).hexdigest()

assert START <= END
assert 0 < MIN_COVERAGE <= 1
assert math.isclose(KURTOSIS_WEIGHT + SKEWNESS_WEIGHT, 1.0)
print(f'Directorio de salida: {OUTPUT_DIR}')

## 2. Descargar y congelar los componentes de Wikipedia

Conserva el HTML de origen y los metadatos de descarga. Mantén los símbolos de Wikipedia para las
respuestas de los estudiantes; sustituye puntos por guiones únicamente en las consultas de precios a Yahoo
(por ejemplo, `BRK.B` → `BRK-B`). La existencia de distintas clases de acciones implica que la tabla
no tiene por qué contener exactamente 500 valores. La instantánea registra la tabla realmente obtenida,
no la composición histórica del índice.


In [ ]:
html_path = OUTPUT_DIR / 'wikipedia_constituents.html'
universe_metadata_path = OUTPUT_DIR / 'universe_source.json'
if not html_path.exists():
    response = requests.get(
        WIKIPEDIA_URL,
        headers={'User-Agent': 'SP500TailRiskEducationalNotebook/1.0'},
        timeout=60,
    )
    response.raise_for_status()
    tables = pd.read_html(StringIO(response.text), attrs={'id': 'constituents'})
    if len(tables) != 1 or not {'Symbol', 'Security'}.issubset(tables[0].columns):
        raise RuntimeError('No se pudo identificar la tabla de componentes de Wikipedia.')
    html_path.write_text(response.text, encoding='utf-8')
    universe_metadata_path.write_text(json.dumps({
        'requested_url': WIKIPEDIA_URL,
        'resolved_url': response.url,
        'retrieved_at_utc': utc_now(),
        'html_sha256': file_hash(html_path),
    }, indent=2), encoding='utf-8')

universe_source = json.loads(universe_metadata_path.read_text(encoding='utf-8'))
if file_hash(html_path) != universe_source['html_sha256']:
    raise RuntimeError('El HTML congelado de Wikipedia ha cambiado.')
constituents = pd.read_html(
    StringIO(html_path.read_text(encoding='utf-8')), attrs={'id': 'constituents'}
)[0]
universe = constituents[['Symbol', 'Security', 'GICS Sector']].rename(columns={
    'Symbol': 'ticker', 'Security': 'company', 'GICS Sector': 'sector'
})
universe['ticker'] = universe['ticker'].astype(str).str.strip().str.upper()
universe['yahoo_ticker'] = universe['ticker'].str.replace('.', '-', regex=False)
universe = universe.sort_values('ticker').reset_index(drop=True)
if universe['ticker'].duplicated().any() or universe['yahoo_ticker'].duplicated().any():
    raise ValueError('Hay símbolos duplicados en Wikipedia o correspondencias ambiguas con símbolos de Yahoo.')
if not 450 <= len(universe) <= 550:
    raise ValueError('Tamaño del universo inesperado; revisa la instantánea de Wikipedia.')
universe.to_csv(OUTPUT_DIR / 'universe.csv', index=False)
print(f'Universo congelado: {len(universe)} valores')
print(f"Descargado: {universe_source['retrieved_at_utc']}")
display(universe.head(10))

## 3. Definir el denominador de sesiones bursátiles

Utiliza el calendario bursátil XNYS, compartido por las acciones cotizadas en Estados Unidos para este ejercicio.
Excluye fines de semana, festivos y cierres excepcionales de la bolsa. Las sesiones reducidas cuentan
como un día bursátil. Utilizar un calendario bursátil independiente evita que posibles interrupciones
del proveedor reduzcan artificialmente el denominador de cobertura.

Descarga también la sesión inmediatamente anterior únicamente para calcular la primera rentabilidad
del periodo. La fecha final está incluida; la fecha final de la API de Yahoo es exclusiva.


In [ ]:
calendar = xcals.get_calendar('XNYS')
sessions = pd.DatetimeIndex(calendar.sessions_in_range(START, END)).tz_localize(None)
if sessions.empty:
    raise ValueError('No hay sesiones bursátiles en el periodo solicitado.')
previous_session = pd.Timestamp(calendar.previous_session(sessions[0])).tz_localize(None)
price_sessions = pd.DatetimeIndex([previous_session]).append(sessions)
expected_sessions = len(sessions)
minimum_valid_returns = math.ceil(MIN_COVERAGE * expected_sessions)
download_start = previous_session.strftime('%Y-%m-%d')
download_end = (END + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
pd.Series(sessions, name='session').to_csv(OUTPUT_DIR / 'trading_sessions.csv', index=False)
print(f'Sesiones esperadas: {expected_sessions}')
print(f'Mínimo de rentabilidades diarias válidas: {minimum_valid_returns}')
print(f'Fecha del precio previo al periodo: {previous_session.date()}')

## 4. Descargar precios diarios de cierre ajustado con caché y registro de reintentos

`auto_adjust=False` solicita explícitamente el campo `Adj Close` del proveedor; nunca se sustituye
por el cierre sin ajustar (`Close`). Las descargas se realizan en lotes pequeños y con un número limitado
de reintentos. Cada símbolo descargado correctamente se almacena individualmente en caché junto con
sus metadatos de descarga y un checksum.

Los historiales parciales se conservan y se evalúan mediante la regla del 90%; no se rellenan artificialmente.

Los símbolos sin un historial utilizable se registran como **fallos de descarga**, por separado de los casos
con cobertura insuficiente o asimetría no negativa. El notebook puede generar un ranking diagnóstico
aunque existan fallos de descarga, pero lo etiquetará como provisional. Resuelve estos fallos antes de
importar las puntuaciones de referencia en Moodle. Establece `RETRY_FAILED_DOWNLOADS=True`
para volver a intentarlo en una ejecución posterior.


In [ ]:
def extract_adjusted_close(raw: pd.DataFrame, symbols: list[str]) -> pd.DataFrame:
    if raw is None or raw.empty:
        return pd.DataFrame(columns=symbols, dtype=float)
    if isinstance(raw.columns, pd.MultiIndex):
        if 'Adj Close' not in raw.columns.get_level_values(0):
            return pd.DataFrame(columns=symbols, dtype=float)
        panel = raw.xs('Adj Close', axis=1, level=0).copy()
    elif len(symbols) == 1 and 'Adj Close' in raw.columns:
        panel = raw[['Adj Close']].rename(columns={'Adj Close': symbols[0]})
    else:
        return pd.DataFrame(columns=symbols, dtype=float)
    panel.index = pd.DatetimeIndex(panel.index).tz_localize(None).normalize()
    if panel.index.has_duplicates:
        raise ValueError('El proveedor ha devuelto fechas diarias duplicadas.')
    return panel.sort_index().reindex(columns=symbols).apply(pd.to_numeric, errors='coerce')

def download_batch(symbols: list[str]) -> pd.DataFrame:
    raw = yf.download(
        symbols, start=download_start, end=download_end, interval='1d',
        auto_adjust=False, back_adjust=False, repair=False, keepna=True,
        actions=False, group_by='column', threads=4, progress=False,
        ignore_tz=True, timeout=30,
    )
    return extract_adjusted_close(raw, symbols)

metadata_path = OUTPUT_DIR / 'price_download_metadata.json'
download_metadata = json.loads(metadata_path.read_text(encoding='utf-8')) if metadata_path.exists() else {}
request_signature = {'start': download_start, 'end_exclusive': download_end,
                     'interval': '1d', 'auto_adjust': False, 'repair': False}
prices_by_ticker = {}
pending = []
for row in universe.itertuples(index=False):
    cache_path = CACHE_DIR / f'{row.yahoo_ticker}.csv'
    record = download_metadata.get(row.ticker)
    if record is not None and record['request'] != request_signature:
        raise ValueError('Los parámetros de descarga de la caché son distintos; elige un nuevo directorio de salida.')
    if cache_path.exists():
        if record is None or record.get('sha256') != file_hash(cache_path):
            raise ValueError(f'Caché de precios no verificada para {row.ticker}.')
        cached = pd.read_csv(cache_path, index_col='date', parse_dates=True)
        prices_by_ticker[row.ticker] = cached['adj_close']
    elif record and record['status'] in {'failed', 'ineligible_listing_history'} and not RETRY_FAILED_DOWNLOADS:
        prices_by_ticker[row.ticker] = pd.Series(dtype=float)
    else:
        pending.append(row)

print(f'Símbolos en caché: {len(prices_by_ticker)}; símbolos por descargar: {len(pending)}', flush=True)
for offset in range(0, len(pending), BATCH_SIZE):
    remaining = pending[offset:offset + BATCH_SIZE]
    errors = {}
    for attempt in range(1, DOWNLOAD_ATTEMPTS + 1):
        if not remaining:
            break
        symbols = [row.yahoo_ticker for row in remaining]
        try:
            panel = download_batch(symbols)
        except Exception as exc:
            panel = pd.DataFrame(columns=symbols, dtype=float)
            errors.update({row.ticker: f'{type(exc).__name__}: {exc}' for row in remaining})
        failed = []
        for row in remaining:
            series = panel[row.yahoo_ticker] if row.yahoo_ticker in panel else pd.Series(dtype=float)
            usable = series.reindex(price_sessions)
            if not (np.isfinite(usable) & (usable > 0)).any():
                errors.setdefault(row.ticker, 'No hay precios Adjusted Close positivos y finitos en las sesiones solicitadas.')
                failed.append(row)
                continue
            series = series.rename('adj_close')
            series.index.name = 'date'
            cache_path = CACHE_DIR / f'{row.yahoo_ticker}.csv'
            series.to_csv(cache_path, float_format='%.17g')
            prices_by_ticker[row.ticker] = pd.read_csv(
                cache_path, index_col='date', parse_dates=True
            )['adj_close']
            download_metadata[row.ticker] = {
                'status': 'ok', 'yahoo_ticker': row.yahoo_ticker,
                'retrieved_at_utc': utc_now(), 'request': request_signature,
                'sha256': file_hash(cache_path), 'nonmissing_rows': int(series.notna().sum()),
            }
        remaining = failed
        if remaining and attempt < DOWNLOAD_ATTEMPTS:
            time.sleep(2 * attempt)
    for row in remaining:
        prices_by_ticker[row.ticker] = pd.Series(dtype=float)
        download_metadata[row.ticker] = {
            'status': 'failed', 'yahoo_ticker': row.yahoo_ticker,
            'retrieved_at_utc': utc_now(), 'request': request_signature,
            'error': errors.get(row.ticker, 'Fallo de descarga no resuelto.'),
        }
    metadata_path.write_text(json.dumps(download_metadata, indent=2), encoding='utf-8')
    print(f'Procesados {min(offset + BATCH_SIZE, len(pending))}/{len(pending)} símbolos nuevos', flush=True)
    time.sleep(0.5)

# Resuelve solicitudes fallidas solo cuando una fecha de cotización documentada demuestra la inelegibilidad.
# No enlaces historiales de valores predecesores a través de fusiones o spin-offs.
listing_evidence = {
    'FDXF': {
        'first_regular_trading_date': '2026-06-01',
        'source': 'https://investor.fedex.com/news-and-events/investor-news/investor-news-details/2026/FedEx-Completes-Spin-Off-of-FedEx-Freight/default.aspx',
    },
    'HONA': {
        'first_regular_trading_date': '2026-06-29',
        'source': 'https://www.honeywell.com/us/en/news/press-releases/2026/06/honeywell-technologies-launches-independent-pure-play-automation-company-following-honeywell-aerospace-spin-off',
    },
    'PSKY': {
        'first_regular_trading_date': '2025-08-07',
        'source': 'https://www.paramount.com/press/skydance-media-and-paramount-global-complete-merger-creating-next-generation-media-company',
    },
}
for ticker, evidence in listing_evidence.items():
    record = download_metadata.get(ticker)
    if record is None or record['status'] not in {'failed', 'ineligible_listing_history'}:
        continue
    listed = pd.Timestamp(evidence['first_regular_trading_date'])
    # Cota superior conservadora: se cuenta incluso el primer día de cotización como rentabilidad válida.
    possible_sessions = int((sessions >= listed).sum())
    if possible_sessions < minimum_valid_returns:
        record.update({
            'status': 'ineligible_listing_history', 'listing_evidence': evidence,
            'maximum_possible_coverage': possible_sessions / expected_sessions,
        })
metadata_path.write_text(json.dumps(download_metadata, indent=2), encoding='utf-8')
pd.DataFrame([
    {'ticker': ticker, **record['listing_evidence'],
     'maximum_possible_coverage': record['maximum_possible_coverage']}
    for ticker, record in download_metadata.items() if record['status'] == 'ineligible_listing_history'
]).to_csv(OUTPUT_DIR / 'confirmed_listing_exclusions.csv', index=False)

adjusted_close = pd.DataFrame(prices_by_ticker).reindex(
    index=price_sessions, columns=universe['ticker'].tolist()
)
download_failures = pd.DataFrame([
    {'ticker': ticker, **record}
    for ticker, record in download_metadata.items() if record['status'] == 'failed'
], columns=['ticker', 'status', 'yahoo_ticker', 'retrieved_at_utc', 'request', 'error'])
adjusted_close.loc[START:END].to_csv(OUTPUT_DIR / 'adjusted_close_daily.csv', float_format='%.17g')
adjusted_close.to_csv(OUTPUT_DIR / 'adjusted_close_with_seed.csv', float_format='%.17g')
download_failures.to_csv(OUTPUT_DIR / 'download_failures.csv', index=False)
READY_FOR_FREEZE = download_failures.empty
print(f'Fallos de descarga no resueltos: {len(download_failures)}')
print(f'Listo para auditoría estadística antes de congelar: {READY_FOR_FREEZE}')
if not download_failures.empty:
    display(download_failures[['ticker', 'error']])

## 5. Rentabilidades entre sesiones consecutivas y elegibilidad

Reindexa los precios sobre la cuadrícula completa de sesiones **antes** de calcular

$r_t=\log(P_t)-\log(P_{t-1})$.

Los precios ausentes, no finitos o no positivos invalidan las rentabilidades afectadas. Un precio ausente
invalida tanto su propia rentabilidad como la de la sesión siguiente. No se realiza *forward fill*,
interpolación ni se enlazan huecos de varias sesiones.

FDXF y HONA comenzaron a cotizar regularmente en 2026; PSKY es un nuevo valor posterior a una fusión
que comenzó a cotizar en agosto de 2025. Los comunicados de los emisores, enlazados en los metadatos
de descarga y en `confirmed_listing_exclusions.csv`, permiten establecer una cobertura máxima posible
inferior al 90%. No se enlazan historiales de valores predecesores. Para estas solicitudes de precios fallidas,
tener cero rentabilidades observadas es un resultado de la descarga, no una afirmación de que no hubo
negociación; el límite superior documentado de cobertura es el que determina de forma independiente
la exclusión.

La cobertura es el número de **rentabilidades** válidas dividido entre todas las sesiones programadas
del periodo de análisis, no la proporción de precios no ausentes. La fecha inicial utilizada únicamente
para calcular la primera rentabilidad se excluye del denominador de cobertura. El umbral incluye
exactamente el 90%.

Se utilizan `skew(..., bias=False)` y `kurtosis(..., fisher=True, bias=False)`; esta última devuelve
el exceso de curtosis (cero para una población normal). Las muestras con varianza cero y las que tienen
menos de cuatro observaciones válidas no tienen métricas de competición definidas. La elegibilidad exige,
además, una asimetría estrictamente negativa.


In [ ]:
def calculate_returns(prices: pd.DataFrame, analysis_sessions: pd.DatetimeIndex) -> pd.DataFrame:
    clean = prices.where(np.isfinite(prices) & (prices > 0))
    return np.log(clean).diff().reindex(analysis_sessions).replace([np.inf, -np.inf], np.nan)

def calculate_metrics(returns: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ticker in returns.columns:
        values = returns[ticker].dropna().to_numpy(dtype=float)
        n_valid = len(values)
        defined = n_valid >= 4 and np.ptp(values) > 0
        rows.append({
            'ticker': ticker, 'n_valid_returns': n_valid,
            'expected_sessions': len(returns), 'coverage': n_valid / len(returns),
            'skewness': float(skew(values, bias=False)) if defined else np.nan,
            'excess_kurtosis': float(kurtosis(values, fisher=True, bias=False)) if defined else np.nan,
            'daily_volatility': float(np.std(values, ddof=1)) if n_valid >= 2 else np.nan,
            'minimum_daily_return': float(values.min()) if n_valid else np.nan,
            'maximum_daily_return': float(values.max()) if n_valid else np.nan,
        })
    return pd.DataFrame(rows)

returns = calculate_returns(adjusted_close, sessions)
metrics = universe.merge(calculate_metrics(returns), on='ticker', validate='one_to_one')
metrics['download_status'] = metrics['ticker'].map(lambda ticker: download_metadata[ticker]['status'])
metrics['maximum_possible_coverage_from_listing'] = metrics['ticker'].map(
    lambda ticker: download_metadata[ticker].get('maximum_possible_coverage', np.nan)
)
metrics['coverage_ok'] = metrics['n_valid_returns'] >= minimum_valid_returns
metrics['moments_defined'] = np.isfinite(metrics['skewness']) & np.isfinite(metrics['excess_kurtosis'])
metrics['negative_skewness'] = metrics['skewness'] < 0
metrics['eligible'] = (
    metrics['download_status'].eq('ok') & metrics['coverage_ok']
    & metrics['moments_defined'] & metrics['negative_skewness']
)

def exclusion_reasons(row: pd.Series) -> str:
    reasons = []
    if row['download_status'] == 'ineligible_listing_history':
        reasons.append('historial_cotizacion_documentado_no_alcanza_90_por_ciento')
    elif row['download_status'] != 'ok':
        reasons.append('fallo_descarga_no_resuelto')
    if not row['coverage_ok']:
        reasons.append('cobertura_rentabilidades_validas_inferior_90_por_ciento')
    if not row['moments_defined']:
        reasons.append('momentos_no_definidos')
    elif not row['negative_skewness']:
        reasons.append('asimetria_no_estrictamente_negativa')
    return '; '.join(reasons)

metrics['exclusion_reasons'] = metrics.apply(exclusion_reasons, axis=1)
returns.to_csv(OUTPUT_DIR / 'daily_log_returns.csv', float_format='%.17g')
print(f'Universo: {len(metrics)} | Elegibles: {metrics.eligible.sum()} | Excluidos: {(~metrics.eligible).sum()}')
display(metrics.groupby(['download_status', 'coverage_ok', 'negative_skewness']).size().rename('stocks'))

## 6. Puntuación combinada equiponderada y clasificación determinista

Entre las acciones elegibles se utilizan los rangos medios de pandas divididos por el tamaño de la muestra
elegible:

$P(x_i)=\operatorname{rank}_{average}(x_i)/N$.

Una curtosis mayor y una asimetría más negativa producen rangos más altos. Las observaciones empatadas
reciben el mismo rango medio.

$C_i=0.5P(K_i)+0.5P(-S_i)$ y $G_i=10C_i/\max_j C_j$.

Se asigna una nota de 10 a todos los ganadores empatados. Los empates de posición utilizan ranking de
competición (1, 1, 3); el orden alfabético se utiliza únicamente para mostrar los resultados.
Estas son las notas de referencia con precisión completa, **todavía no redondeadas a las fracciones de
respuesta aceptadas por Moodle**. Este notebook no genera XML de Moodle. Resuelve todos los fallos
de descarga antes de utilizar la clasificación para puntuar.


In [ ]:
def rank_eligible(table: pd.DataFrame) -> pd.DataFrame:
    ranked = table.loc[table['eligible']].copy()
    if ranked.empty:
        raise ValueError('No hay acciones elegibles; revisa la auditoría de descarga y cobertura.')
    ranked['kurtosis_percentile'] = ranked['excess_kurtosis'].rank(method='average', pct=True)
    ranked['negative_skewness_percentile'] = (-ranked['skewness']).rank(method='average', pct=True)
    ranked['combined_score'] = (
        KURTOSIS_WEIGHT * ranked['kurtosis_percentile']
        + SKEWNESS_WEIGHT * ranked['negative_skewness_percentile']
    )
    best_score = ranked['combined_score'].max()
    ranked['distance_to_best_score'] = best_score - ranked['combined_score']
    ranked['grade_out_of_10'] = 10 * ranked['combined_score'] / best_score
    ranked['competition_rank'] = ranked['combined_score'].rank(ascending=False, method='min').astype(int)
    return ranked.sort_values(['combined_score', 'ticker'], ascending=[False, True]).reset_index(drop=True)

leaderboard = rank_eligible(metrics)
leaderboard['reference_status'] = 'audit_required' if READY_FOR_FREEZE else 'provisional_download_failures'
display(leaderboard[['competition_rank', 'ticker', 'company', 'coverage', 'excess_kurtosis',
                     'skewness', 'combined_score', 'distance_to_best_score', 'grade_out_of_10']].head(20))

## 7. Verificación y auditoría de rentabilidades extremas

Estas comprobaciones verifican los límites metodológicos: los precios ausentes no deben generar rentabilidades
que atraviesen huecos, una cobertura exactamente igual al 90% debe ser válida, una asimetría igual a cero
debe ser excluida, los empates deben conservar la misma puntuación y las entradas no deben contener fechas
de rentabilidad fuera de la ventana de análisis.

Las mayores rentabilidades en valor absoluto se exportan para realizar una revisión manual de operaciones
corporativas y calidad de datos. Conserva los valores extremos válidos; corrige errores reales de datos
únicamente mediante un conjunto de datos de referencia alternativo explícitamente documentado y recalcula
todas las puntuaciones de forma coherente.


In [ ]:
toy_dates = pd.date_range('2024-01-01', periods=5)
toy_prices = pd.DataFrame({'X': [100, 110, np.nan, 120, 126]}, index=toy_dates)
toy_returns = calculate_returns(toy_prices, toy_dates[1:])['X']
assert np.isclose(toy_returns.iloc[0], np.log(1.1))
assert toy_returns.iloc[1:3].isna().all()
assert np.isclose(toy_returns.iloc[3], np.log(1.05))
assert 90 >= math.ceil(0.90 * 100) and not 89 >= math.ceil(0.90 * 100)
assert not (0.0 < 0)
constant_metrics = calculate_metrics(pd.DataFrame({'X': np.zeros(10)}))
assert constant_metrics[['skewness', 'excess_kurtosis']].isna().all().all()
tied = rank_eligible(pd.DataFrame({
    'ticker': ['A', 'B'], 'eligible': [True, True],
    'excess_kurtosis': [5.0, 5.0], 'skewness': [-1.0, -1.0],
}))
assert tied['combined_score'].nunique() == 1 and tied['grade_out_of_10'].eq(10).all()
assert returns.index.equals(sessions)
assert metrics['n_valid_returns'].between(0, expected_sessions).all()
assert leaderboard['coverage'].ge(MIN_COVERAGE).all()
assert leaderboard['skewness'].lt(0).all()
assert np.isfinite(leaderboard[['skewness', 'excess_kurtosis', 'combined_score']]).all().all()
assert np.isclose(leaderboard['grade_out_of_10'].max(), 10)
assert len(metrics) == len(universe)
# Verifica de forma independiente las estimaciones de momentos corregidas por sesgo de SciPy para el líder.
leader_ticker = leaderboard.loc[0, 'ticker']
observed = returns[leader_ticker].dropna().to_numpy()
n = len(observed)
centered = observed - observed.mean()
m2, m3, m4 = [np.mean(centered ** order) for order in (2, 3, 4)]
manual_skewness = np.sqrt(n * (n - 1)) / (n - 2) * m3 / m2 ** 1.5
raw_excess = m4 / m2 ** 2 - 3
manual_kurtosis = (n - 1) / ((n - 2) * (n - 3)) * ((n + 1) * raw_excess + 6)
assert np.isclose(manual_skewness, leaderboard.loc[0, 'skewness'], rtol=1e-10)
assert np.isclose(manual_kurtosis, leaderboard.loc[0, 'excess_kurtosis'], rtol=1e-10)
extreme_returns = returns.rename_axis('date').stack().rename('log_return').reset_index()
extreme_returns.columns = ['date', 'ticker', 'log_return']
extreme_returns['absolute_log_return'] = extreme_returns['log_return'].abs()
extreme_returns = extreme_returns.sort_values(
    ['absolute_log_return', 'ticker', 'date'], ascending=[False, True, True]
)
extreme_returns.head(100).to_csv(OUTPUT_DIR / 'extreme_returns_audit.csv', index=False)
print('Comprobaciones metodológicas superadas. Revisa la auditoría de rentabilidades extremas antes de congelar.')
display(extreme_returns.head(15))

## 8. Exportar tablas de referencia, feedback y manifiesto de reproducibilidad

Las exportaciones incluyen el universo completo con los motivos de exclusión, los rankings de activos elegibles,
precios y rentabilidades diarios, fallos de descarga y una tabla de preparación para Moodle. Los fallos de
descarga reciben un estado no resuelto en lugar de una nota cero inventada. Los demás activos no elegibles
reciben un cero, tal como establecen las reglas de la competición.

Conserva este directorio después de la revisión. Si se requiere reproducibilidad exacta entre distintos
proveedores de datos, los estudiantes deberían recibir los datos de precios de referencia, pero no las tablas
con las respuestas. El manifiesto registra fuentes, fechas, definiciones, versiones de dependencias y hashes
de los datos de entrada.


In [ ]:
score_columns = ['ticker', 'competition_rank', 'kurtosis_percentile', 'negative_skewness_percentile',
                 'combined_score', 'distance_to_best_score', 'grade_out_of_10']
reference = metrics.merge(leaderboard[score_columns], on='ticker', how='left', validate='one_to_one')
reference.loc[~reference['eligible'] & reference['download_status'].ne('failed'), 'grade_out_of_10'] = 0.0
reference['reference_status'] = 'audit_required' if READY_FOR_FREEZE else 'provisional_download_failures'

def make_feedback(row: pd.Series) -> str:
    if row['download_status'] == 'ineligible_listing_history':
        return (
            'Activo no elegible: su historial de cotización documentado no puede alcanzar el 90% de cobertura. '
            f"Cobertura máxima posible: {row['maximum_possible_coverage_from_listing']:.2%}. Nota: 0/10."
        )
    if row['download_status'] != 'ok':
        return 'Datos de referencia no resueltos. No utilices todavía esta respuesta para calificar.'
    if not row['eligible']:
        return f"Activo no elegible. Motivo: {row['exclusion_reasons']}. Nota: 0/10."
    return (
        f"Exceso de curtosis: {row['excess_kurtosis']:.6f}. "
        f"Asimetría: {row['skewness']:.6f}. "
        f"Cobertura de rentabilidades válidas: {row['coverage']:.2%}. "
        f"Puntuación combinada: {row['combined_score']:.6f}. "
        f"Distancia a la mejor puntuación: {row['distance_to_best_score']:.6f}. "
        f"Nota de referencia: {row['grade_out_of_10']:.4f}/10."
    )

reference['feedback'] = reference.apply(make_feedback, axis=1)
reference.to_csv(OUTPUT_DIR / 'all_assets_metrics.csv', index=False, float_format='%.17g')
leaderboard.to_csv(OUTPUT_DIR / 'eligible_leaderboard.csv', index=False, float_format='%.17g')
reference.loc[~reference['eligible']].to_csv(OUTPUT_DIR / 'excluded_assets.csv', index=False)
reference[['ticker', 'eligible', 'grade_out_of_10', 'feedback', 'reference_status']].to_csv(
    OUTPUT_DIR / 'moodle_reference.csv', index=False
)
packages = ['numpy', 'pandas', 'scipy', 'requests', 'lxml', 'yfinance', 'exchange-calendars']
package_versions = {name: version(name) for name in packages}
manifest = {
    'generated_at_utc': utc_now(), 'python': platform.python_version(),
    'analysis_start': str(START.date()), 'analysis_end_inclusive': str(END.date()),
    'previous_price_session': str(previous_session.date()),
    'competition_open': '2026-10-08T14:00:00+02:00',
    'competition_close': '2026-10-12T23:59:00+02:00', 'competition_timezone': 'Europe/Madrid',
    'maximum_attempts': 5, 'calendar': 'XNYS', 'expected_sessions': expected_sessions,
    'minimum_valid_returns': minimum_valid_returns, 'minimum_coverage': MIN_COVERAGE,
    'prices': 'Yahoo Finance Adj Close, auto_adjust=False, repair=False',
    'returns': 'diferencias logarítmicas naturales de precios ajustados entre sesiones bursátiles consecutivas',
    'skewness': 'scipy.stats.skew(bias=False)',
    'kurtosis': 'scipy.stats.kurtosis(fisher=True, bias=False)',
    'eligibility': 'coverage >= 0.90; skewness < 0; momentos finitos; descarga correcta',
    'percentile_definition': 'pandas rank(method=average, pct=True) entre activos elegibles',
    'weights': {'kurtosis': KURTOSIS_WEIGHT, 'negative_skewness': SKEWNESS_WEIGHT},
    'grade_definition': '10 * combined_score / maximum_eligible_combined_score',
    'universe_size': len(universe), 'eligible_assets': int(metrics['eligible'].sum()),
    'download_failures': len(download_failures),
    'listing_evidence': listing_evidence,
    'status': 'audit_required' if READY_FOR_FREEZE else 'provisional_download_failures',
    'universe_source': universe_source, 'packages': package_versions,
    'price_cache_hashes': {ticker: record.get('sha256') for ticker, record in download_metadata.items()},
    'file_hashes': {p.name: file_hash(p) for p in sorted(OUTPUT_DIR.glob('*.csv'))},
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'requirements-lock.txt').write_text(
    '\n'.join(f'{name}=={v}' for name, v in package_versions.items()) + '\n', encoding='utf-8'
)
display(reference.loc[reference['eligible'], ['ticker', 'grade_out_of_10', 'feedback']].sort_values(
    ['grade_out_of_10', 'ticker'], ascending=[False, True]
).head(5))
print(f'Todos los archivos de referencia se han exportado a {OUTPUT_DIR}')
print(f"Estado de referencia: {manifest['status']}")

## Fuentes e interpretación

- [Tabla de componentes de Wikipedia](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies)
- [Parámetros de descarga de yfinance](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html)
- [Calendarios bursátiles y definición de sesiones](https://github.com/gerrymanoim/exchange_calendars)
- [Asimetría en SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.skew.html)
- [Curtosis en SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kurtosis.html)

Este notebook establece las respuestas *in-sample* de la competición. No valida una predicción de riesgo.
Cualquier afirmación posterior sobre capacidad predictiva del riesgo requiere un periodo *out-of-sample*
independiente, un universo *point-in-time* y un análisis adecuado de incertidumbre. No interpretes un ranking
percentil empírico como una probabilidad de pérdida ni como una prueba de normalidad.
